In [1]:
#importing the sqlite3 module

import sqlite3

In [2]:
# Connecting to DB and creating a cursor object
conn = sqlite3.connect('wordnet.db')#creating a database connection
cursor = conn.cursor()
conn.close()

In [3]:
#parsing the file, and print all the lines, but leaving the ones that start with a space

def parse_file(filepath):
    with open(filepath, 'r') as file:
        for line in file:
            if line.startswith(" "):
                continue
            # Further processing can go here
            print(line)  # Example of processing

# Call the function with the correct path format
parse_file('/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/dict/data.adj')


00001740 00 a 01 able 0 005 = 05207437 n 0000 = 05624029 n 0000 + 05624029 n 0101 + 05207437 n 0101 ! 00002098 a 0101 | (usually followed by `to') having the necessary means or skill or know-how or authority to do something; "able to swim"; "she was able to program her computer"; "we were at last able to buy a car"; "able to get a grant for the project"  

00002098 00 a 01 unable 0 002 = 05207437 n 0000 ! 00001740 a 0101 | (usually followed by `to') not having the necessary means or skill or know-how; "unable to get to town without a car"; "unable to obtain funds"  

00002312 00 a 02 abaxial 0 dorsal 4 002 ;c 06047178 n 0000 ! 00002527 a 0101 | facing away from the axis of an organ or organism; "the abaxial surface of a leaf is the underside or side facing away from the stem"  

00002527 00 a 02 adaxial 0 ventral 4 002 ;c 06047178 n 0000 ! 00002312 a 0101 | nearest to or facing toward the axis of an organ or organism; "the upper side of a leaf is known as the adaxial surface"  

000027

In [4]:
#if length is less then 2 then leaving them, and extracting the word, definition and example from the file
def parse_file(filepath):
    with open(filepath, 'r') as file:
        for line in file:
            parts = line.split("|")
            if len(parts) < 2:
                continue
            word_info = parts[0].strip()
            definition = parts[1].strip()
            example = parts[2].strip() if len(parts) > 2 else None
            if example:
                examples = example.split(",")  # Split examples by commas
                examples = [ex.strip() for ex in examples]  # Clean up spaces around examples
            else:
                examples = None
            # Example of how to use the extracted information
            print(f"Word: {word_info}, Definition: {definition}, Example: {example}")

# Example function call with a file path
parse_file('/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/dict/data.adj')


Word: 00001740 00 a 01 able 0 005 = 05207437 n 0000 = 05624029 n 0000 + 05624029 n 0101 + 05207437 n 0101 ! 00002098 a 0101, Definition: (usually followed by `to') having the necessary means or skill or know-how or authority to do something; "able to swim"; "she was able to program her computer"; "we were at last able to buy a car"; "able to get a grant for the project", Example: None
Word: 00002098 00 a 01 unable 0 002 = 05207437 n 0000 ! 00001740 a 0101, Definition: (usually followed by `to') not having the necessary means or skill or know-how; "unable to get to town without a car"; "unable to obtain funds", Example: None
Word: 00002312 00 a 02 abaxial 0 dorsal 4 002 ;c 06047178 n 0000 ! 00002527 a 0101, Definition: facing away from the axis of an organ or organism; "the abaxial surface of a leaf is the underside or side facing away from the stem", Example: None
Word: 00002527 00 a 02 adaxial 0 ventral 4 002 ;c 06047178 n 0000 ! 00002312 a 0101, Definition: nearest to or facing towar

In [10]:
cursor.execute('''
INSERT INTO words(word, defenition, example)
VALUES(?,?,?)
''', (word_info, defenition, example))

NameError: name 'word_info' is not defined

In [14]:
conn.commit()  # Save changes to the database
conn.close()  # Close the connection


ProgrammingError: Cannot operate on a closed database.

In [6]:
import sqlite3

# Function to parse a WordNet data file and insert its content into the database
def parse_and_insert(filepath, cursor):
    with open(filepath, 'r') as file:
        for line in file:
            parts = line.split("|")
            if len(parts) < 2:
                continue
            word_info = parts[0].strip()
            definition = parts[1].strip()
            example = parts[2].strip() if len(parts) > 2 else None
            cursor.execute('''
                INSERT INTO words(word, definition, example)
                VALUES(?,?,?)
            ''', (word_info, definition, example))

# Function to handle database connection and manage parsing of multiple files
def populate_database(db_path, filepaths):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Create a table if it doesn't exist
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS words (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            word TEXT NOT NULL,
            definition TEXT NOT NULL,
            example TEXT
        )
    ''')
    
    # Parse each file and insert data
    for filepath in filepaths:
        parse_and_insert(filepath, cursor)
    
    conn.commit()  # Save changes to the database
    conn.close()   # Close the connection

# Paths to the WordNet data files
filepaths = [
    '/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/dict/data.adj',
    '/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/dict/data.noun',
    '/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/dict/data.adv',
    '/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/dict/data.verb'
]

# Path to the SQLite database
db_path = '/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/wordnet.db'

# Populate the database with data from the files
populate_database(db_path, filepaths)


In [7]:
import sqlite3

# Function to fetch and display a few rows from the database
def view_data(db_path, limit=10):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Fetch a few rows from the words table
    cursor.execute('SELECT * FROM words LIMIT ?', (limit,))
    rows = cursor.fetchall()
    
    for row in rows:
        print(f"ID: {row[0]}, Word: {row[1]}, Definition: {row[2]}, Example: {row[3]}")
    
    conn.close()  # Close the connection

# Path to the SQLite database
db_path = '/Users/surisettivamsikrishna/Downloads/Vamsi Pc/CODES/Mark2/SARAS/Saras_-Smart-Define/DataBase/wordnet.db'

# View data
view_data(db_path)


ID: 1, Word: 00001740 00 a 01 able 0 005 = 05207437 n 0000 = 05624029 n 0000 + 05624029 n 0101 + 05207437 n 0101 ! 00002098 a 0101, Definition: (usually followed by `to') having the necessary means or skill or know-how or authority to do something; "able to swim"; "she was able to program her computer"; "we were at last able to buy a car"; "able to get a grant for the project", Example: None
ID: 2, Word: 00002098 00 a 01 unable 0 002 = 05207437 n 0000 ! 00001740 a 0101, Definition: (usually followed by `to') not having the necessary means or skill or know-how; "unable to get to town without a car"; "unable to obtain funds", Example: None
ID: 3, Word: 00002312 00 a 02 abaxial 0 dorsal 4 002 ;c 06047178 n 0000 ! 00002527 a 0101, Definition: facing away from the axis of an organ or organism; "the abaxial surface of a leaf is the underside or side facing away from the stem", Example: None
ID: 4, Word: 00002527 00 a 02 adaxial 0 ventral 4 002 ;c 06047178 n 0000 ! 00002312 a 0101, Definition